# Problem Statement

Climate change misinformation spreads widely through social media, blogs, and news articles. False claims such as “climate change is a hoax” can influence public opinion and hinder climate action.

The goal of this project is to build an AI/NLP model that automatically detects whether a climate-related statement is:

- Misinformation
- Factual

The system processes textual statements using Natural Language Processing (NLP) techniques and classifies them using Machine Learning models.

The project demonstrates the complete NLP pipeline:

> Dataset → Text preprocessing → Feature extraction → Model training → Evaluation → Prediction.

Datasets for this project typically contain text statements and labels such as misinformation or factual.

# Datasets

A suitable dataset for this project is the Climate Change Tweets Dataset, which contains climate-related claims and their truth labels.

```s
| text                           | label          |
| ------------------------------ | -------------- |
| Climate change is fake         | misinformation |
| Rising CO2 causes warming      | factual        |
| Global warming stopped in 1998 | misinformation |
```

Database File Name: climate_data.csv

## Installing Packages

In [3]:
!pip install pandas numpy scikit-learn nltk matplotlib seaborn kagglehub --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Importing Libraries & Downloading NLTK Resources

In [ ]:
import pandas as pd
import numpy as np
import nltk
import string
import matplotlib.pyplot as plt
import seaborn as sns

'''
    Download required NLTK resources in one loop
    punkt: Tokenizer models for splitting text into words
    stopwords: List of common words to remove from text
    wordnet: Lexical database for English used for lemmatization & synonymnym lookup
    omw-1.4: Open Multilingual Wordnet for extended lemmatization support
'''
for resource in ("punkt", "stopwords", "wordnet", "omw-1.4"):
    nltk.download(resource)

from nltk.tokenize import word_tokenize # Tokenizer for splitting text into words
from nltk.corpus import stopwords # List of common words to remove from text
from nltk.stem import WordNetLemmatizer # Lemmatizer for reducing words to their base form

from sklearn.feature_extraction.text import TfidfVectorizer # vectorizer to convert text to numerical features
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB # Naive Bayes classifier for text classification
from sklearn.linear_model import LogisticRegression # Logistic Regression classifier for text classification
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/codespace/nltk_data...


## Load Dataset

In [ ]:
data = pd.read_csv("climate_data.csv")

print("Dataset Shape:", data.shape)
print(data.head())

: 

## Preprocessing

## Explore Dataset

In [ ]:
print("\nClass Distribution:")
print(data['label'].value_counts())

# Plot label distribution
sns.countplot(x='label', data=data)
plt.title("Distribution of Climate Statements")
plt.show()

## NLP Preprocessing with NLTK

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Tokenization
    tokens = word_tokenize(text)

    # Remove punctuation
    tokens = [word for word in tokens if word not in string.punctuation]

    # Stopword removal
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Join tokens back
    return " ".join(tokens)

## Preprocessing

In [ ]:
data['clean_text'] = data['text'].apply(preprocess_text)
print(data[['text','clean_text']].head())

## Word Frequency Visualization

In [ ]:
from collections import Counter

all_words = " ".join(data['clean_text']).split()

word_freq = Counter(all_words).most_common(20)

words = [w[0] for w in word_freq]
counts = [w[1] for w in word_freq]

plt.figure(figsize=(10,5))
plt.bar(words, counts)
plt.xticks(rotation=45)
plt.title("Most Frequent Words")
plt.show()

## Convert Text to TF-IDF (Term Frequency-Inverse Document Frequency)

In [ ]:
vectorizer = TfidfVectorizer() # vectorizer to convert text to numerical features

X = vectorizer.fit_transform(data['clean_text'])
y = data['label']

## Data Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train Models

In [ ]:
# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

## Model Evaluation

In [ ]:
def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label='misinformation')
    rec = recall_score(y_test, y_pred, pos_label='misinformation')
    f1 = f1_score(y_test, y_pred, pos_label='misinformation')

    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1 Score:", f1)

    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


print("\nNaive Bayes Performance")
evaluate_model(nb_model, X_test, y_test)

print("\nLogistic Regression Performance")
evaluate_model(lr_model, X_test, y_test)

## Model Comparision

In [ ]:
models = ['Naive Bayes', 'Logistic Regression']
accuracies = [
    accuracy_score(y_test, nb_model.predict(X_test)),
    accuracy_score(y_test, lr_model.predict(X_test))
]

plt.bar(models, accuracies)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.show()

## Prediction Function

In [ ]:
def predict_statement(statement, model):

    cleaned = preprocess_text(statement)

    vector = vectorizer.transform([cleaned])

    prediction = model.predict(vector)[0]

    print("Statement:", statement)
    print("Prediction:", prediction)

## Sample Predictions

In [ ]:
predict_statement(
    "Climate change is a hoax created by scientists",
    lr_model
)

predict_statement(
    "Carbon dioxide emissions increase global temperatures",
    lr_model
)

predict_statement(
    "Renewable energy reduces greenhouse gases",
    lr_model
)